In [1]:
import os
import glob
import pickle
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

root = "/kaggle/input/datasets/subihanbiswas/pkl-files"

run_files = sorted(glob.glob(os.path.join(root, "raw_predictions_*.pkl")))

print(f"Pooling across {len(run_files)} runs")
print(*run_files, sep="\n")

Pooling across 4 runs
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_original_run.pkl
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_seed101.pkl
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_seed202.pkl
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_seed303.pkl


In [2]:
degradation_conditions = {
    "All present":        None,
    "Fingerprint missing": ["fp"],
    "Iris missing":        ["iris"],
    "Voice missing":       ["voice"],
    "FP + Iris missing":   ["fp", "iris"],
    "FP + Voice missing":  ["fp", "voice"],
    "Iris + Voice missing":["iris", "voice"],
}


In [3]:
all_runs = [pickle.load(open(f, "rb")) for f in run_files]

print(f"\n{'Condition':25s} {'n_pooled':>10s} {'p-value':>10s}   Interpretation")
print("-" * 75)

for cond in degradation_conditions:
    pooled_adaptive_preds = np.concatenate([r["adaptive"][cond]["preds"] for r in all_runs])
    pooled_adaptive_labels = np.concatenate([r["adaptive"][cond]["labels"] for r in all_runs])
    pooled_naive_preds = np.concatenate([r["naive"][cond]["preds"] for r in all_runs])
    pooled_naive_labels = np.concatenate([r["naive"][cond]["labels"] for r in all_runs])

    assert np.array_equal(pooled_adaptive_labels, pooled_naive_labels)

    a_correct = (pooled_adaptive_preds == pooled_adaptive_labels)
    n_correct = (pooled_naive_preds == pooled_naive_labels)

    both_correct = int(np.sum(a_correct & n_correct))
    only_adaptive = int(np.sum(a_correct & ~n_correct))
    only_naive = int(np.sum(~a_correct & n_correct))
    both_wrong = int(np.sum(~a_correct & ~n_correct))

    table = [[both_correct, only_adaptive], [only_naive, both_wrong]]
    result = mcnemar(table, exact=(only_adaptive + only_naive < 25))

    sig = "significant (p<0.05)" if result.pvalue < 0.05 else "not significant"
    print(f"{cond:25s} {len(pooled_adaptive_labels):10d} {result.pvalue:10.4f}   {sig}")


Condition                   n_pooled    p-value   Interpretation
---------------------------------------------------------------------------
All present                    13056     0.5000   not significant
Fingerprint missing            13056     0.3750   not significant
Iris missing                   13056     0.2317   not significant
Voice missing                  13056     0.0784   not significant
FP + Iris missing              13056     0.3326   not significant
FP + Voice missing             13056     0.1273   not significant
Iris + Voice missing           13056     0.0488   significant (p<0.05)


In [4]:
for cond in degradation_conditions:
    pooled_adaptive_preds = np.concatenate([r["adaptive"][cond]["preds"] for r in all_runs])
    pooled_adaptive_labels = np.concatenate([r["adaptive"][cond]["labels"] for r in all_runs])
    pooled_naive_preds = np.concatenate([r["naive"][cond]["preds"] for r in all_runs])

    a_correct = (pooled_adaptive_preds == pooled_adaptive_labels)
    n_correct = (pooled_naive_preds == pooled_adaptive_labels)

    pooled_adaptive_acc = a_correct.mean()
    pooled_naive_acc = n_correct.mean()

    print(f"{cond:25s} Adaptive: {pooled_adaptive_acc:.4f}  Naive: {pooled_naive_acc:.4f}  "
          f"Favors: {'Adaptive' if pooled_adaptive_acc > pooled_naive_acc else 'Naive'}")

All present               Adaptive: 1.0000  Naive: 0.9998  Favors: Adaptive
Fingerprint missing       Adaptive: 0.9996  Naive: 0.9994  Favors: Adaptive
Iris missing              Adaptive: 0.9934  Naive: 0.9923  Favors: Adaptive
Voice missing             Adaptive: 0.9992  Naive: 0.9985  Favors: Adaptive
FP + Iris missing         Adaptive: 0.8637  Naive: 0.8611  Favors: Adaptive
FP + Voice missing        Adaptive: 0.9930  Naive: 0.9939  Favors: Naive
Iris + Voice missing      Adaptive: 0.8799  Naive: 0.8867  Favors: Naive


In [5]:
root = "/kaggle/input/datasets/subihanbiswas/pkl-file2"

run_files = sorted(glob.glob(os.path.join(root, "raw_predictions_*.pkl")))

print(f"Pooling across {len(run_files)} runs")
print(*run_files, sep="\n")

Pooling across 6 runs
/kaggle/input/datasets/subihanbiswas/pkl-file2/raw_predictions_original_run.pkl
/kaggle/input/datasets/subihanbiswas/pkl-file2/raw_predictions_seed101.pkl
/kaggle/input/datasets/subihanbiswas/pkl-file2/raw_predictions_seed202.pkl
/kaggle/input/datasets/subihanbiswas/pkl-file2/raw_predictions_seed303.pkl
/kaggle/input/datasets/subihanbiswas/pkl-file2/raw_predictions_seed404.pkl
/kaggle/input/datasets/subihanbiswas/pkl-file2/raw_predictions_seed505.pkl


In [6]:
all_runs = [pickle.load(open(f, "rb")) for f in run_files]

print(f"\n{'Condition':25s} {'n_pooled':>10s} {'p-value':>10s}   Interpretation")
print("-" * 75)

for cond in degradation_conditions:
    pooled_adaptive_preds = np.concatenate([r["adaptive"][cond]["preds"] for r in all_runs])
    pooled_adaptive_labels = np.concatenate([r["adaptive"][cond]["labels"] for r in all_runs])
    pooled_naive_preds = np.concatenate([r["naive"][cond]["preds"] for r in all_runs])
    pooled_naive_labels = np.concatenate([r["naive"][cond]["labels"] for r in all_runs])

    assert np.array_equal(pooled_adaptive_labels, pooled_naive_labels)

    a_correct = (pooled_adaptive_preds == pooled_adaptive_labels)
    n_correct = (pooled_naive_preds == pooled_naive_labels)

    both_correct = int(np.sum(a_correct & n_correct))
    only_adaptive = int(np.sum(a_correct & ~n_correct))
    only_naive = int(np.sum(~a_correct & n_correct))
    both_wrong = int(np.sum(~a_correct & ~n_correct))

    table = [[both_correct, only_adaptive], [only_naive, both_wrong]]
    result = mcnemar(table, exact=(only_adaptive + only_naive < 25))

    sig = "significant (p<0.05)" if result.pvalue < 0.05 else "not significant"
    print(f"{cond:25s} {len(pooled_adaptive_labels):10d} {result.pvalue:10.4f}   {sig}")


Condition                   n_pooled    p-value   Interpretation
---------------------------------------------------------------------------
All present                    19584     0.5000   not significant
Fingerprint missing            19584     1.0000   not significant
Iris missing                   19584     0.6058   not significant
Voice missing                  19584     0.4414   not significant
FP + Iris missing              19584     0.5783   not significant
FP + Voice missing             19584     0.0117   significant (p<0.05)
Iris + Voice missing           19584     0.0000   significant (p<0.05)


In [8]:
for cond in degradation_conditions:
    pooled_adaptive_preds = np.concatenate([r["adaptive"][cond]["preds"] for r in all_runs])
    pooled_adaptive_labels = np.concatenate([r["adaptive"][cond]["labels"] for r in all_runs])
    pooled_naive_preds = np.concatenate([r["naive"][cond]["preds"] for r in all_runs])

    a_correct = (pooled_adaptive_preds == pooled_adaptive_labels)
    n_correct = (pooled_naive_preds == pooled_adaptive_labels)

    pooled_adaptive_acc = a_correct.mean()
    pooled_naive_acc = n_correct.mean()

    print(f"{cond:25s} Adaptive: {pooled_adaptive_acc:.4f}  Naive: {pooled_naive_acc:.4f}  "
          f"Favors: {'Adaptive' if pooled_adaptive_acc > pooled_naive_acc else 'Naive'}")

All present               Adaptive: 1.0000  Naive: 0.9999  Favors: Adaptive
Fingerprint missing       Adaptive: 0.9995  Naive: 0.9994  Favors: Adaptive
Iris missing              Adaptive: 0.9938  Naive: 0.9934  Favors: Adaptive
Voice missing             Adaptive: 0.9990  Naive: 0.9987  Favors: Adaptive
FP + Iris missing         Adaptive: 0.8631  Naive: 0.8619  Favors: Adaptive
FP + Voice missing        Adaptive: 0.9935  Naive: 0.9945  Favors: Naive
Iris + Voice missing      Adaptive: 0.8782  Naive: 0.8937  Favors: Naive
